[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThanhDatVN/AI_journey/blob/main/notebooks/02_mathematics_for_ml/06_probability.ipynb)

# Bài 06 — Xác suất cho Machine Learning

Lý thuyết xác suất là một trong những nền tảng quan trọng nhất của Machine Learning: dữ liệu có nhiễu, model đưa ra **xác suất** (spam với xác suất 0.93), và nhiều thuật toán được xây dựng trực tiếp từ xác suất (Naive Bayes, Logistic Regression, GMM, ...). Hiểu rõ xác suất giúp bạn đi sâu hơn vào những thuật toán phức tạp.

**Nội dung**
1. Khái niệm xác suất: không gian mẫu, biến cố, tiên đề, hoán vị, tổ hợp *(từ AIVN)*
2. Xác suất có điều kiện, công thức Bayes, độc lập *(từ AIVN, bổ sung ví dụ & mô phỏng)*
3. *(bổ sung)* Biến ngẫu nhiên, phân phối, kỳ vọng, phương sai, hiệp phương sai
4. *(bổ sung)* Luật số lớn, định lý giới hạn trung tâm, Monte Carlo

In [ ]:
# ⚙️ Setup — chạy được cả local lẫn Google Colab (local: đã `pip install -e .` thì bỏ qua)
import sys
if "google.colab" in sys.modules:
    !test -d /content/ml-course || git clone -q --depth 1 https://github.com/ThanhDatVN/AI_journey /content/ml-course
    !pip install -q -e /content/ml-course
    sys.path.insert(0, "/content/ml-course/src")
    %cd /content/ml-course/notebooks/02_mathematics_for_ml

In [ ]:
import math
from itertools import combinations, permutations

import matplotlib.pyplot as plt
import numpy as np
from scipy import stats

from ml_course.visualization import set_style

set_style()
rng = np.random.default_rng(0)

## 1. Khái niệm xác suất

### 1.1 Không gian mẫu (Sample Space)

Không gian mẫu, thường ký hiệu $S$ hay $\Omega$, của một phép thử ngẫu nhiên là tập hợp **tất cả các kết quả có thể xảy ra**. Ví dụ: tung đồng xu → $\{\text{ngửa}, \text{sấp}\}$; tung xúc xắc 6 mặt → $\{1, 2, 3, 4, 5, 6\}$.

Bất kỳ tập con nào của không gian mẫu được gọi là một **biến cố (event)**.

### 1.2 Phép thử và biến cố

Phép thử là một thử nghiệm cho kết quả là một biến cố. Ví dụ, tung xúc xắc là một phép thử; "ra mặt chẵn" $= \{2, 4, 6\}$ là một biến cố. Các tính chất của đại số tập hợp áp dụng được cho biến cố:

- Giao hoán: $A \cup B = B \cup A$, $A \cap B = B \cap A$
- Kết hợp: $A \cup (B \cup C)=(A \cup B) \cup C$, $A \cap (B \cap C)=(A \cap B) \cap C$
- Phân phối: $A \cap (B \cup C)=(A \cap B) \cup (A \cap C)$, $A\cup (B \cap C)=(A \cup B)\cap(A \cup C)$
- Phần bù (De Morgan): $\overline{\overline{A}}=A$, $\overline{A \cup B} = \overline{A} \cap \overline{B}$, $\overline{A \cap B} = \overline{A} \cup \overline{B}$

### 1.3 Tiên đề xác suất (Kolmogorov)

Với mỗi biến cố $E$, ký hiệu $P(E)$ là xác suất của $E$.

**Tiên đề 1:** mọi xác suất đều nằm trong khoảng $[0, 1]$.

<img src="images/xac-suat/1.png" style="width:45%;height:45%;">

**Tiên đề 2:** xác suất của toàn bộ không gian mẫu bằng 1 — không có kết quả nào nằm ngoài không gian mẫu.

<img src="images/xac-suat/2.png" style="width:45%;height:45%;">

**Tiên đề 3:** với dãy biến cố đôi một không giao nhau $E_1, E_2, \dots$, xác suất của hợp bằng tổng các xác suất:

<img src="images/xac-suat/3.png" style="width:45%;height:45%;">

Hệ quả hay dùng: $P(\overline A) = 1 - P(A)$ và $P(A \cup B) = P(A) + P(B) - P(A \cap B)$.

### 1.4 Hoán vị (P)

Một hoán vị là một bộ **có thứ tự, không lặp**. Số hoán vị chập $r$ của $n$ phần tử: phần tử đầu có $n$ cách chọn, phần tử thứ hai $(n-1)$ cách, ..., phần tử thứ $r$ có $(n - r + 1)$ cách:

<img src="images/xac-suat/4.png" style="width:45%;height:45%;">

Ví dụ $n = 10$, $r = 3$: $P(10, 3) = 10 \cdot 9 \cdot 8 = 720$.

### 1.5 Tổ hợp (C)

Tổ hợp là cách chọn phần tử **không phân biệt thứ tự**. Ví dụ, chọn 2 trong 3 loại quả {táo, cam, lê} có 3 cách: táo–lê, táo–cam, lê–cam.

<img src="images/xac-suat/5.png" style="width:45%;height:45%;">

**Ghi chú:** với $0 \le r \le n$ ta có $P(n,r) \ge C(n, r)$ (vì $P(n, r) = C(n, r)\cdot r!$).

In [ ]:
print("P(10, 3) =", math.perm(10, 3), "| kiểm tra bằng liệt kê:", len(list(permutations(range(10), 3))))
print("C(10, 3) =", math.comb(10, 3), "| kiểm tra bằng liệt kê:", len(list(combinations(range(10), 3))))
print("Các tổ hợp 2 loại quả:", list(combinations(["táo", "cam", "lê"], 2)))

# Ứng dụng ML: số cặp feature có thể tạo interaction feature (ch.17), số tập con feature (ch.18)
d = 20
print(f"\nVới {d} feature: {math.comb(d, 2)} cặp tương tác, {2 ** d:,} tập con feature có thể chọn")

> 💡 $2^{20}$ ≈ 1 triệu tập con — đó là lý do không thể thử mọi tổ hợp feature (chương 18: feature selection dùng chiến lược tham lam thay vì vét cạn).

## 2. Xác suất có điều kiện

Xác suất có điều kiện là xác suất xảy ra biến cố $A$ khi **biết** biến cố $B$ đã xảy ra:

$$P(A \mid B) = \frac{P(A \cap B)}{P(B)}, \quad P(B) > 0$$

Từ đó có **quy tắc nhân**: $P(A \cap B) = P(A \mid B)P(B) = P(B \mid A)P(A)$.

### Công thức Bayes

$$P(A \mid B) = \frac{P(B \mid A)\,P(A)}{P(B)}, \qquad P(B) = \sum_k P(B \mid A_k)P(A_k)$$

Trong ML: $P(A)$ là **prior** (niềm tin ban đầu), $P(B \mid A)$ là **likelihood**, $P(A \mid B)$ là **posterior** (niềm tin sau khi thấy dữ liệu). Naive Bayes (chương 09) dùng trực tiếp công thức này.

### Ví dụ kinh điển *(bổ sung)*: xét nghiệm bệnh hiếm

Một bệnh có tỉ lệ mắc 1%. Xét nghiệm có độ nhạy 95% (người bệnh → dương tính) và tỉ lệ dương tính giả 5%. Một người có kết quả **dương tính**. Xác suất người đó thật sự mắc bệnh?

In [ ]:
p_disease = 0.01
p_pos_given_disease = 0.95
p_pos_given_healthy = 0.05

p_pos = p_pos_given_disease * p_disease + p_pos_given_healthy * (1 - p_disease)
p_disease_given_pos = p_pos_given_disease * p_disease / p_pos
print(f"P(bệnh | dương tính) = {p_disease_given_pos:.3f}")

# Kiểm chứng bằng mô phỏng 1 triệu người
n = 1_000_000
sick = rng.random(n) < p_disease
positive = np.where(sick, rng.random(n) < p_pos_given_disease, rng.random(n) < p_pos_given_healthy)
print(f"Mô phỏng: {sick[positive].mean():.3f}")

Chỉ ~16%! Phần lớn người dương tính là **dương tính giả** vì bệnh quá hiếm — đây là **base rate fallacy**. Bài học cho ML: với dữ liệu **mất cân bằng** (gian lận thẻ, bệnh hiếm), một model "chính xác 95%" vẫn có thể cho phần lớn cảnh báo sai (chương 14, 19).

### 3. Biến cố độc lập

$A$ và $B$ độc lập khi và chỉ khi $P(A \cap B) = P(A)P(B)$, tương đương $P(A \mid B) = P(A)$: biết $B$ không thay đổi niềm tin về $A$.

**Độc lập có điều kiện** — $P(A \cap B \mid C) = P(A \mid C)P(B \mid C)$ — là giả định "ngây thơ" của **Naive Bayes**.

## 3. Biến ngẫu nhiên và phân phối *(bổ sung)*

**Biến ngẫu nhiên** $X$ gán một số cho mỗi kết quả của phép thử.
- **Rời rạc**: hàm khối xác suất (PMF) $P(X = x)$, $\sum_x P(X=x) = 1$.
- **Liên tục**: hàm mật độ (PDF) $p(x)$, $P(a \le X \le b) = \int_a^b p(x)dx$. Chú ý $p(x)$ có thể > 1!
- Hàm phân phối tích luỹ (CDF): $F(x) = P(X \le x)$.

Các phân phối gặp nhiều nhất trong ML:

| Phân phối | Mô hình hoá | Gặp ở |
|---|---|---|
| Bernoulli($p$) | một lần thử có/không | nhãn nhị phân, logistic regression |
| Binomial($n, p$) | số lần thành công trong $n$ lần thử | A/B testing |
| Categorical | một trong $K$ lớp | softmax, phân loại nhiều lớp |
| Poisson($\lambda$) | số sự kiện trong khoảng thời gian | số cuộc gọi hỗ trợ, đếm từ |
| Uniform($a, b$) | mọi giá trị như nhau | khởi tạo trọng số, random search |
| Normal $\mathcal N(\mu, \sigma^2)$ | tổng của nhiều yếu tố nhỏ | nhiễu đo đạc, linear regression |

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 7))
k = np.arange(0, 21)
axes[0, 0].bar([0, 1], stats.bernoulli(0.3).pmf([0, 1]))
axes[0, 0].set_title("Bernoulli(p=0.3)")
axes[0, 1].bar(k, stats.binom(20, 0.3).pmf(k))
axes[0, 1].set_title("Binomial(n=20, p=0.3)")
axes[0, 2].bar(k, stats.poisson(4).pmf(k))
axes[0, 2].set_title("Poisson(λ=4)")
x = np.linspace(-5, 5, 400)
axes[1, 0].plot(x, stats.uniform(-2, 4).pdf(x))
axes[1, 0].set_title("Uniform(−2, 2)")
for mu, s in [(0, 1), (0, 2), (2, 0.5)]:
    axes[1, 1].plot(x, stats.norm(mu, s).pdf(x), label=f"μ={mu}, σ={s}")
axes[1, 1].legend(fontsize=8)
axes[1, 1].set_title("Normal — chú ý pdf có thể > 1")
axes[1, 2].plot(x, stats.norm(0, 1).cdf(x))
axes[1, 2].set_title("CDF của N(0, 1)")
fig.tight_layout()
plt.show()

### Kỳ vọng, phương sai, hiệp phương sai

$$\mathbb E[X] = \sum_x x\,P(X=x) \;\;\text{(hoặc } \textstyle\int x\,p(x)dx), \qquad \operatorname{Var}(X) = \mathbb E[(X - \mathbb E X)^2] = \mathbb E[X^2] - (\mathbb E X)^2$$

$$\operatorname{Cov}(X, Y) = \mathbb E[(X - \mathbb EX)(Y - \mathbb EY)], \qquad \rho_{XY} = \frac{\operatorname{Cov}(X,Y)}{\sigma_X\sigma_Y} \in [-1, 1]$$

Tính chất cần nhớ:
- Tuyến tính: $\mathbb E[aX + bY] = a\mathbb E X + b\mathbb E Y$ (luôn đúng, kể cả khi không độc lập).
- $\operatorname{Var}(aX + b) = a^2\operatorname{Var}(X)$.
- $\operatorname{Var}(X + Y) = \operatorname{Var}X + \operatorname{Var}Y + 2\operatorname{Cov}(X, Y)$ → nếu độc lập, phương sai cộng được. Đây là lý do **trung bình nhiều model độc lập giảm phương sai** (bagging, chương 11).
- Độc lập ⇒ Cov = 0, nhưng Cov = 0 **không** suy ra độc lập.

In [ ]:
dice = np.arange(1, 7)
E = np.mean(dice)
Var = np.mean(dice**2) - E**2
print(f"Xúc xắc: E = {E}, Var = {Var:.4f}")
samples = rng.integers(1, 7, 100_000)
print(f"Mô phỏng: E ≈ {samples.mean():.4f}, Var ≈ {samples.var():.4f}")

# Cov = 0 nhưng phụ thuộc hoàn toàn: Y = X²
X = rng.normal(size=100_000)
Y = X**2
print(f"\nX ~ N(0,1), Y = X²: Cov(X, Y) ≈ {np.cov(X, Y)[0, 1]:.4f} — nhưng Y phụ thuộc hoàn toàn vào X!")

## 4. Luật số lớn, định lý giới hạn trung tâm, Monte Carlo *(bổ sung)*

**Luật số lớn (LLN):** trung bình mẫu $\bar X_n$ hội tụ về $\mathbb E[X]$ khi $n \to \infty$.

**Định lý giới hạn trung tâm (CLT):** với $n$ đủ lớn, $\bar X_n$ xấp xỉ phân phối chuẩn $\mathcal N(\mu, \sigma^2/n)$ **bất kể** phân phối gốc → sai số của trung bình giảm theo $1/\sqrt n$. Đây là cơ sở của khoảng tin cậy (bài 08).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
draws = rng.exponential(scale=2.0, size=5000)          # phân phối lệch, E = 2
running_mean = np.cumsum(draws) / np.arange(1, len(draws) + 1)
axes[0].plot(running_mean)
axes[0].axhline(2.0, color="red", ls="--", label="E[X] = 2")
axes[0].set_xscale("log")
axes[0].set_xlabel("n")
axes[0].set_title("Luật số lớn: trung bình mẫu → kỳ vọng")
axes[0].legend()

for n in [1, 5, 30]:
    means = rng.exponential(2.0, size=(20_000, n)).mean(axis=1)
    axes[1].hist(means, bins=80, density=True, histtype="step", lw=2, label=f"n = {n}")
axes[1].set_title("CLT: phân phối của trung bình mẫu → hình chuông")
axes[1].legend()
plt.show()

**Monte Carlo:** ước lượng một kỳ vọng/tích phân bằng trung bình của các mẫu ngẫu nhiên. Ví dụ: xác suất tổng 2 xúc xắc ≥ 10 (đáp án chính xác $6/36$):

In [ ]:
for n in [100, 10_000, 1_000_000]:
    rolls = rng.integers(1, 7, size=(n, 2)).sum(axis=1)
    print(f"n={n:>9,}: ước lượng {np.mean(rolls >= 10):.4f}  (đúng: {6 / 36:.4f})")

## 🧠 Kiểm tra nhanh

1. Có bao nhiêu cách chọn 3 feature từ 10 feature (không quan tâm thứ tự)?
2. Trong ví dụ xét nghiệm, nếu tỉ lệ mắc bệnh là 10% thì $P(\text{bệnh} \mid +)$ thay đổi thế nào? Vì sao?
3. Hai biến có hiệp phương sai bằng 0 — chúng có độc lập không?
4. Sai số chuẩn của trung bình mẫu thay đổi thế nào khi tăng $n$ lên 4 lần?

## Tổng kết

- Xác suất: không gian mẫu, biến cố, 3 tiên đề; đếm bằng hoán vị/tổ hợp.
- Bayes: posterior ∝ likelihood × prior — cẩn thận base rate với dữ liệu hiếm.
- Biến ngẫu nhiên & phân phối; kỳ vọng tuyến tính; phương sai của tổng phụ thuộc hiệp phương sai.
- LLN + CLT: trung bình mẫu ổn định dần với sai số $\propto 1/\sqrt n$; Monte Carlo dựa trên điều này.

## Tài liệu tham khảo

[1] Khái niệm lý thuyết xác suất — Wikipedia · [2] [Xác suất — Khái niệm cơ bản — Hai's Blog](https://dominhhai.github.io/vi/2017/10/what-is-prob/) · [3] Seeing Theory — https://seeing-theory.brown.edu (trực quan hoá tuyệt vời) · [4] Blitzstein & Hwang — *Introduction to Probability* (miễn phí online)

---
<sub>Bài gốc: *Xác suất cơ bản* — [AIVN-Machine-Learning](https://github.com/thanhhff/AIVN-Machine-Learning), Tuần 1 (MIT License). Đã bổ sung: code đếm hoán vị/tổ hợp, ví dụ Bayes + mô phỏng, biến ngẫu nhiên & phân phối, kỳ vọng/phương sai/hiệp phương sai, LLN, CLT, Monte Carlo.</sub>